In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv
from os import environ
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy.engine.base import Engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import CatBoostEncoder
from sklearn.compose import ColumnTransformer
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
    median_absolute_error
)

# Подключение к БД

In [2]:
load_dotenv(find_dotenv())

True

In [3]:
DB_USER = environ['DB_USER']
DB_HOST = environ['DB_HOST']
DB_PORT = environ['DB_PORT']
DB_NAME = environ['DB_NAME']
DB_PASSWORD = environ['DB_PASSWORD']

In [4]:
db_conn_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

In [5]:
conn = create_engine(db_conn_url)

# Загрузка датасета из БД

In [6]:
sql = "select * from buildings_flats"

In [7]:
data = pd.read_sql(sql, conn)

Закрываем подключение к БД:

In [8]:
conn.dispose()

# EDA

## Анализ загруженых данных

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100928 entries, 0 to 100927
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   building_id        100928 non-null  int64  
 1   flat_id            100928 non-null  int64  
 2   build_year         100928 non-null  int64  
 3   building_type_int  100928 non-null  int64  
 4   latitude           100928 non-null  float64
 5   longitude          100928 non-null  float64
 6   ceiling_height     100928 non-null  float64
 7   flats_count        100928 non-null  int64  
 8   floors_total       100928 non-null  int64  
 9   has_elevator       100928 non-null  bool   
 10  floor              100928 non-null  int64  
 11  kitchen_area       100928 non-null  float64
 12  living_area        100928 non-null  float64
 13  rooms              100928 non-null  int64  
 14  is_apartment       100928 non-null  bool   
 15  studio             100928 non-null  bool   
 16  to

Настройка параметров отображения для работы с большими данными:

In [10]:
pd.options.display.max_columns = 100
pd.options.display.max_rows = 64 

Основные статистические характеристики числовых данных:

In [11]:
data.describe()

,building_id,flat_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,floor,kitchen_area,living_area,rooms,total_area,price
count,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,100928.000000,1.009280e+05
mean,13235.552344,71595.130202,1984.260621,3.532449,55.730954,37.597841,2.686337,242.093562,13.084090,6.763227,8.646471,30.936057,1.949816,51.883511,1.183833e+07
std,6358.539275,41424.026863,19.208103,1.402605,0.107924,0.157480,0.129421,169.010672,5.080863,4.508283,2.280058,11.544195,0.818618,16.237297,4.532862e+06
min,4.000000,0.000000,1902.000000,0.000000,55.211460,36.864372,2.400000,1.000000,1.000000,1.000000,4.000000,8.000000,1.000000,16.000000,3.000000e+06
25%,8336.000000,33718.250000,1969.000000,4.000000,55.648075,37.494484,2.640000,112.000000,9.000000,3.000000,6.500000,20.000000,1.000000,38.500000,8.500000e+06
50%,12985.000000,73257.000000,1980.000000,4.000000,55.718594,37.590549,2.640000,201.000000,12.000000,6.000000,8.600000,30.000000,2.000000,49.000000,1.080000e+07
75%,18433.000000,106921.250000,2002.000000,4.000000,55.817533,37.720547,2.700000,320.000000,17.000000,10.000000,10.000000,38.900002,3.000000,60.799999,1.400000e+07
max,24620.000000,141361.000000,2023.000000,6.000000,56.011032,37.946411,3.000000,4455.000000,29.000000,20.000000,16.500000,75.000000,5.000000,121.000000,2.900000e+07


Количество пропусков:

In [12]:
data.isnull().sum().sort_values(ascending=False)

building_id          0
flat_id              0
total_area           0
studio               0
is_apartment         0
rooms                0
living_area          0
kitchen_area         0
floor                0
has_elevator         0
floors_total         0
flats_count          0
ceiling_height       0
longitude            0
latitude             0
building_type_int    0
build_year           0
price                0
dtype: int64

<u>**Анализ**</u>

**Выброс в flats_count**: Значение 4455 нужно ограничить, например, на уровне 99-го перцентиля.

**Кодировка типа здания**: `building_type_int` можно преобразовать в категориальный тип (Categorical / Object), чтобы модель не пыталась искать линейную зависимость между "типом 1" и "типом 6".

**Пропуски**: Явных пропусков нет.